# Comparação dos modelos com DensGNN (adaptive)

Tabela no formato da referência: um bloco por **modelo**, uma linha por **métrica** e uma coluna por **dataset**. As variantes DensGNN (core) e DensGNN (border) também são incluídas. DensGNN (adaptive) aparece no último bloco e continua sendo a referência das cores.

- **Vermelho**: média melhor que a do DensGNN adaptive no mesmo dataset/métrica.
- **Azul**: média pior que a dele.
- Empates, a própria referência e comparações sem referência disponível ficam sem destaque. Não há negrito para marcar máximos.
- As cores respeitam a direção da métrica: ↑ maior é melhor; ↓ menor é melhor. Quantidade de clusters não tem uma direção de melhora.

O notebook lê resultados salvos e **não executa treinos**. Pode ser executado mesmo quando nenhuma run terminou. As funções de leitura, agregação e formatação estão em `scripts/benchmark_comparison.py`.

## Configurar a tabela

Por padrão, usa os YAMLs atualmente presentes em `configs/core_border_adaptive_densgnn` para descobrir os modelos, datasets, caminhos de resultados e seeds esperadas. Assim, a tabela acompanha as configurações existentes sem fixar a quantidade de datasets ou modelos no código. Os YAMLs `core.yaml` e `border.yaml` fornecem as variantes adicionais. Seus diretórios originais, como `results/cora_core` e `results/cora_border`, são agrupados na coluna Cora sem misturar suas runs com o adaptive. `adaptive.yaml` fica de fora para não duplicar a referência já definida por `densgnn.yaml`; DeepDensGNN continua fora da comparação. Quando um dataset não tem configuração de core/border, essas células mostram `N/A`.

As quatro métricas iniciais seguem a tabela da referência: NMI, ARI, ACC e **F1-macro**. Use `f1_micro` se quiser a outra definição de F1. NMI/ARI/ACC/F1 são apresentados em porcentagem, incluindo o desvio padrão. As demais métricas mantêm suas unidades originais. Para incluir todas, use `TABLE_METRICS = list(comparison.METRIC_DIRECTIONS)`.

A análise consulta somente `results/`. Variantes configuradas sem CSVs nessa pasta aparecem como pendentes.

In [33]:
from pathlib import Path
import sys

from IPython.display import display, HTML, Markdown


def find_repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "benchmark_comparison.py").is_file():
            return candidate
    raise FileNotFoundError("Abra o notebook dentro do repositório.")


ROOT = find_repo_root()
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import importlib
import benchmark_comparison as comparison
importlib.reload(comparison)

CONFIG_DIR = ROOT / "configs/core_border_adaptive_densgnn"
EXPORT_DIR = ROOT / "results/adaptive_benchmark_comparison"
TABLE_METRICS = ["nmi", "ari", "acc", "f1_macro"]
INTERNAL_METRICS = ["silhouette", "calinski_harabasz", "davies_bouldin", "modularity"]
PERCENT_SCORES = True
LATEX_STYLE = "ranked"  # Maior média em negrito; segunda maior distinta sublinhada.

## Ler o estado atual das execuções

Reexecute esta célula e as seguintes para atualizar a tabela enquanto o benchmark continua. Os arquivos são um retrato dos resultados já gravados, sem monitoramento automático.

Somente `status=success` entra nas médias. Para tentativas repetidas, é mantida a última tentativa bem-sucedida por dataset/modelo/seed. Falhas, inclusive OOT, não viram nota zero. CSVs incompletos ou inválidos são registrados em `report.issues` e não interrompem a análise.

In [34]:
snapshot = comparison.build_report(
    CONFIG_DIR,
    ROOT,
    table_metrics=list(dict.fromkeys(TABLE_METRICS + INTERNAL_METRICS)),
    percent=PERCENT_SCORES,
)
# As duas tabelas usam exatamente as mesmas runs e o mesmo instante de leitura.
report = snapshot.for_metrics(TABLE_METRICS, percent=PERCENT_SCORES)
internal_report = snapshot.for_metrics(INTERNAL_METRICS, percent=False)
print(f"Retrato dos resultados: {report.snapshot}")
print(f"Tentativas lidas: {len(report.attempts)}; selecionadas após retries: {len(report.selected)}")
print(f"Execuções bem-sucedidas: {report.audit['n_success'].sum()}")
print(f"Execuções com falha: {report.audit['n_failed'].sum()} (OOT: {report.audit['n_oot'].sum()})")
print(f"Sem resultado terminal: {report.audit['n_pending'].sum()}")
if not report.issues.empty:
    display(Markdown("### Arquivos/linhas ignorados neste retrato"))
    display(report.issues)

Retrato dos resultados: 2026-10-17T08:28:40-03:00
Tentativas lidas: 501; selecionadas após retries: 501
Execuções bem-sucedidas: 475
Execuções com falha: 26 (OOT: 26)
Sem resultado terminal: 109


## Tabela de métricas externas

Cada célula mostra **média ± desvio padrão amostral** (`ddof=1`). O marcador `(n=k/N)` informa que há menos valores válidos do que as N runs previstas. Uma única run não recebe desvio padrão zero: o desvio é omitido.

- `Pendente (0/N)`: nenhuma execução terminou; pode estar na fila ou em andamento.
- `Sem sucesso (0/N)`: há falhas gravadas, mas nenhum resultado válido de sucesso.
- `OOT (0/N)`: todas as N seeds selecionadas terminaram por limite de tempo.
- `Indefinido (0/N)`: há runs bem-sucedidas, mas a métrica é indefinida em todas elas.
- `N/A`: métrica ou variante não configurada, como as supervisionadas no Facebook ou core/border em datasets sem esses YAMLs.

As comparações usam as médias disponíveis, que podem ter quantidades de runs e seeds diferentes; não são testes de significância estatística. Se o DensGNN adaptive ainda não tiver média válida em um dataset/métrica, os outros modelos ficam sem vermelho/azul nessa comparação. As cores podem mudar à medida que novas runs terminam.

In [35]:
display(HTML(
    '<p><span style="color:#b91c1c">Vermelho: melhor que DensGNN adaptive</span> · '
    '<span style="color:#1d4ed8">Azul: pior que DensGNN adaptive</span> · '
    'Sem destaque: empate, referência ou referência indisponível</p>'
))
if PERCENT_SCORES:
    print("NMI, ARI, ACC e F1 em porcentagem; as outras métricas mantêm suas unidades.")
display(report.styled())

NMI, ARI, ACC e F1 em porcentagem; as outras métricas mantêm suas unidades.


## Tabela de métricas internas

Usa os mesmos modelos, datasets e runs da tabela anterior, incluindo DensGNN core/border. As quatro métricas mantêm suas unidades originais:

- Silhouette ↑: maior é melhor.
- Calinski–Harabasz (CH) ↑: maior é melhor.
- Davies–Bouldin (DB) ↓: **menor é melhor**.
- Modularidade ↑: maior é melhor.

As cores continuam relativas ao DensGNN adaptive, com vermelho para melhor e azul para pior. Uma métrica interna indefinida permanece ausente; não recebe zero. Não é necessário terminar o benchmark para gerar esta tabela.

In [36]:
display(internal_report.styled())

## Seleção exploratória: adaptive até a 3ª posição

Esta célula cria um **subconjunto condicionado aos resultados**, separado das tabelas completas. Ela escolhe o maior número de modelos que permite ao DensGNN adaptive ficar até a terceira posição **em pelo menos uma das quatro métricas externas e uma das quatro internas, em cada um dos três datasets**. O mesmo conjunto de modelos é usado em todas as comparações.

A posição é `1 + número de modelos com média estritamente melhor`. Empates com o adaptive não contam como melhores. Davies–Bouldin usa a direção menor é melhor. Havendo vários subconjuntos máximos do mesmo tamanho, o desempate segue a ordem alfabética dos identificadores dos modelos.

Um concorrente precisa ter média válida em todas as condições selecionadas; caso contrário, aparece entre os excluídos por dados insuficientes. Se faltar alguma média do próprio adaptive, a seleção aguarda essas runs, sem eliminar métricas ou datasets silenciosamente. Médias parciais continuam identificadas com `n=k/N`; o subconjunto pode mudar após novas runs.

A auditoria mostra tanto a posição **no subconjunto** quanto **entre todos os modelos com resultados disponíveis**. Esta seleção não substitui as tabelas completas nem representa a posição do adaptive no benchmark inteiro.

A tabela de cobertura identifica quais métricas atendem ao limite. As demais métricas também são exibidas, mesmo quando o adaptive fica abaixo da terceira posição. Isso permite ampliar a comparação sem ocultar suas limitações.

Os arquivos LaTeX deste recorte são exportados sem cores: maior média em negrito e segunda maior média distinta sublinhada, por dataset/métrica. Empates compartilham o destaque; a classificação usa valores antes do arredondamento. Esse destaque segue os maiores valores numéricos, inclusive em Davies–Bouldin, cuja direção de melhora continua sendo menor.

In [37]:
import json
import pandas as pd

SELECTION_DATASETS = ["cora", "citeseer", "polblogs"]
SELECTION_EXTERNAL_METRICS = ["nmi", "ari", "acc", "f1_macro"]
SELECTION_INTERNAL_METRICS = ["silhouette", "calinski_harabasz", "davies_bouldin", "modularity"]
MAX_ADAPTIVE_RANK = 3
SUBSET_EXCLUDED_MODELS = ["coane_topk", "ns4gc", "dgc_efr", "egc_vgae"]  # Mesmas remoções nas tabelas externa e interna.
MIN_TOP_METRICS_PER_GROUP = 1
SELECTION_METRIC_GROUPS = {
    "external": SELECTION_EXTERNAL_METRICS,
    "internal": SELECTION_INTERNAL_METRICS,
}

selection = comparison.select_top_rank_subset(
    snapshot,
    datasets=SELECTION_DATASETS,
    metrics=SELECTION_EXTERNAL_METRICS + SELECTION_INTERNAL_METRICS,
    max_rank=MAX_ADAPTIVE_RANK,
    metric_groups=SELECTION_METRIC_GROUPS,
    min_metrics_per_group=MIN_TOP_METRICS_PER_GROUP,
)

if selection.report is None:
    print("Seleção pendente: o adaptive precisa ter média válida em todas as condições.")
    display(selection.pending_reference)
else:
    print(f"Subconjunto condicionado aos resultados: {len(selection.selected_models)} modelos, incluindo adaptive.")
    display(pd.DataFrame({
        "modelo": selection.selected_models,
        "nome": [comparison.MODEL_NAMES.get(model, model) for model in selection.selected_models],
    }))
    display(Markdown("### Cobertura: métricas em que adaptive atende ao limite por dataset/grupo"))
    display(selection.coverage)
    display(Markdown("### Posições do adaptive: seleção automática antes das remoções manuais"))
    display(selection.ranks)
    display(Markdown("### Modelos excluídos e motivos"))
    display(selection.exclusions)

    subset_external = selection.report.for_metrics(
        SELECTION_EXTERNAL_METRICS, percent=PERCENT_SCORES
    ).without_models(SUBSET_EXCLUDED_MODELS)
    subset_internal = selection.report.for_metrics(
        SELECTION_INTERNAL_METRICS, percent=False
    ).without_models(SUBSET_EXCLUDED_MODELS)
    display(Markdown("### Métricas externas — após as remoções manuais"))
    display(subset_external.styled())
    display(Markdown("### Métricas internas — mesmos modelos da tabela externa"))
    display(subset_internal.styled())

    # Exportações próprias; as tabelas completas permanecem em seus caminhos.
    selection_dir = EXPORT_DIR / f"top{MAX_ADAPTIVE_RANK}_subset"
    comparison.export_report(subset_external, selection_dir / "external", table_label="tab:subset-external", latex_style=LATEX_STYLE)
    comparison.export_report(subset_internal, selection_dir / "internal", table_label="tab:subset-internal", latex_style=LATEX_STYLE)
    selection.coverage.to_csv(selection_dir / "criterion_coverage.csv", index=False)
    selection.ranks.to_csv(selection_dir / "adaptive_ranks.csv", index=False)
    selection.exclusions.to_csv(selection_dir / "excluded_models.csv", index=False)
    (selection_dir / "selection_criteria.json").write_text(json.dumps({
        "description": "Seleção automática máxima, com as mesmas remoções manuais nas duas tabelas; não é o benchmark completo.",
        "snapshot": snapshot.snapshot,
        "datasets": SELECTION_DATASETS,
        "metrics": SELECTION_EXTERNAL_METRICS + SELECTION_INTERNAL_METRICS,
        "max_adaptive_rank": MAX_ADAPTIVE_RANK,
        "metric_groups": SELECTION_METRIC_GROUPS,
        "min_top_metrics_per_group_per_dataset": MIN_TOP_METRICS_PER_GROUP,
        "rank_definition": "1 + número de concorrentes com média estritamente melhor",
        "tie_break": "ordem alfabética dos identificadores dos modelos",
        "selected_models": selection.selected_models,
        "external_excluded_models": SUBSET_EXCLUDED_MODELS,
        "internal_excluded_models": SUBSET_EXCLUDED_MODELS,
        "external_models": subset_external.summary.model.drop_duplicates().tolist(),
        "internal_models": subset_internal.summary.model.drop_duplicates().tolist(),
        "latex_style": LATEX_STYLE,
        "latex_emphasis": "Maior média em negrito; segunda maior média distinta sublinhada, independentemente da direção da métrica.",
    }, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"Exportações do subconjunto: {selection_dir}")

Subconjunto condicionado aos resultados: 17 modelos, incluindo adaptive.


,modelo,nome
0,arga_kmeans,ARGA + KMeans
1,coane,CoANE
2,coane_topk,CoANE top-k
3,dgc_efr,DGC-EFR
4,dgcss,DGCSS
5,dmon,DMoN
6,dsagc,DSAGC
7,egc_vgae,EGC-VGAE
8,gae_kmeans,GAE + KMeans
9,kmeans_attributes,KMeans (attributes)


### Cobertura: métricas em que adaptive atende ao limite por dataset/grupo

,dataset,metric_group,n_metrics,n_within_limit,required,best_rank,qualifying_metrics
0,cora_benchmark_adaptive,external,4,4,1,1,"nmi, ari, acc, f1_macro"
1,cora_benchmark_adaptive,internal,4,4,1,1,"silhouette, calinski_harabasz, davies_bouldin,..."
2,citeseer_benchmark_adaptive,external,4,1,1,1,acc
3,citeseer_benchmark_adaptive,internal,4,1,1,3,silhouette
4,polblogs_benchmark_adaptive,external,4,1,1,3,nmi
5,polblogs_benchmark_adaptive,internal,4,3,1,2,"silhouette, calinski_harabasz, davies_bouldin"


### Posições do adaptive: seleção automática antes das remoções manuais

...

### Modelos excluídos e motivos

,model,reason,details
0,densgnn_border,Dados insuficientes,cora_benchmark_adaptive/nmi; cora_benchmark_ad...
1,densgnn_core,Dados insuficientes,cora_benchmark_adaptive/nmi; cora_benchmark_ad...
2,dgcluster,Limite de posição no subconjunto máximo,
3,scgc,Limite de posição no subconjunto máximo,


### Métricas externas — após as remoções manuais

### Métricas internas — mesmos modelos da tabela externa

Exportações do subconjunto: /home/gmessias/NodeClusteringAlgorithms/results/adaptive_benchmark_comparison/top3_subset


## Acompanhar runs incompletas, falhas e referência

A contagem abaixo separa resultados terminais, sucessos, falhas, OOT e pendências. `n_pending` inclui tanto runs não iniciadas quanto runs em andamento; o CSV sozinho não permite distinguir esses dois casos. Métricas indefinidas podem ter menos valores válidos do que o número de sucessos.

In [38]:
reference = report.audit.loc[report.audit["model"] == comparison.REFERENCE_MODEL]
display(Markdown("### Disponibilidade do DensGNN adaptive"))
display(reference[["dataset", "expected_runs", "n_success", "n_failed", "n_oot", "n_pending", "state"]])
reference_metrics = snapshot.summary.loc[snapshot.summary["model"] == comparison.REFERENCE_MODEL]
if not reference_metrics.empty:
    display(reference_metrics.pivot(index="dataset", columns="metric", values="n_valid"))

progress = report.audit.copy()
progress["progresso"] = [
    f"{row.n_success}/{row.expected_runs} sucessos; {row.n_oot} OOT; {row.n_pending} pendentes"
    for row in progress.itertuples(index=False)
]
progress_table = progress.pivot(index="model", columns="dataset", values="progresso")
model_order = list(report.table.index.get_level_values("Modelo").unique())
progress_table.index = [comparison.MODEL_NAMES.get(model, model) for model in progress_table.index]
display(Markdown("### Progresso por modelo/dataset"))
display(progress_table.reindex(model_order).fillna("N/A"))

problems = report.audit.loc[(report.audit["n_failed"] > 0) | (report.audit["n_pending"] > 0)]
if not problems.empty:
    display(Markdown("### Grupos com falhas ou resultados pendentes"))
    display(problems)

### Disponibilidade do DensGNN adaptive

,dataset,expected_runs,n_success,n_failed,n_oot,n_pending,state
5,citeseer_benchmark_adaptive,5,5,0,0,0,Concluído
26,cora_benchmark_adaptive,5,5,0,0,0,Concluído
47,coraml_benchmark_adaptive,5,5,0,0,0,Concluído
66,facebook_benchmark_adaptive,5,5,0,0,0,Concluído
87,polblogs_benchmark_adaptive,5,5,0,0,0,Concluído
106,wiki_benchmark_adaptive,5,0,1,1,4,Pendente/em execução


metric,acc,ari,calinski_harabasz,davies_bouldin,f1_macro,modularity,nmi,silhouette
dataset,,,,,,,,
citeseer_benchmark_adaptive,5,5,5,5,5,5,5,5
cora_benchmark_adaptive,5,5,5,5,5,5,5,5
coraml_benchmark_adaptive,5,5,5,5,5,5,5,5
facebook_benchmark_adaptive,0,0,5,5,0,5,0,5
polblogs_benchmark_adaptive,5,5,5,5,5,5,5,5
wiki_benchmark_adaptive,0,0,0,0,0,0,0,0


### Progresso por modelo/dataset

dataset,citeseer_benchmark_adaptive,cora_benchmark_adaptive,coraml_benchmark_adaptive,facebook_benchmark_adaptive,polblogs_benchmark_adaptive,wiki_benchmark_adaptive
ARGA + KMeans,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes
CoANE,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 5 OOT; 0 pendentes
CoANE top-k,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 5 OOT; 0 pendentes
DGC-EFR,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 0 OOT; 5 pendentes
DGCluster,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 0 OOT; 5 pendentes
DGCSS,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 0 OOT; 5 pendentes
DMoN,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 0 OOT; 5 pendentes
DSAGC,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 0 OOT; 5 pendentes
EGC-VGAE,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 5 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 0 OOT; 5 pendentes
GAE + KMeans,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,5/5 sucessos; 0 OOT; 0 pendentes,0/5 sucessos; 0 OOT; 5 pendentes


### Grupos com falhas ou resultados pendentes

,dataset,model,expected_runs,source_dataset,source_model,result_path,n_completed,n_success,n_failed,n_oot,n_pending,state
22,cora_benchmark_adaptive,densgnn_border,5,cora_border,densgnn,/home/gmessias/NodeClusteringAlgorithms/result...,0,0,0,0,5,Pendente/em execução
25,cora_benchmark_adaptive,densgnn_core,5,cora_core,densgnn,/home/gmessias/NodeClusteringAlgorithms/result...,0,0,0,0,5,Pendente/em execução
43,coraml_benchmark_adaptive,densgnn_border,5,coraml_border,densgnn,/home/gmessias/NodeClusteringAlgorithms/result...,0,0,0,0,5,Pendente/em execução
46,coraml_benchmark_adaptive,densgnn_core,5,coraml_core,densgnn,/home/gmessias/NodeClusteringAlgorithms/result...,0,0,0,0,5,Pendente/em execução
72,facebook_benchmark_adaptive,egc_vgae,5,facebook_benchmark_adaptive,egc_vgae,/home/gmessias/NodeClusteringAlgorithms/result...,5,0,5,5,0,Concluído com falhas
77,facebook_benchmark_adaptive,sanece,5,facebook_benchmark_adaptive,sanece,/home/gmessias/NodeClusteringAlgorithms/result...,5,0,5,5,0,Concluído com falhas
79,facebook_benchmark_adaptive,soli,5,facebook_benchmark_adaptive,soli,/home/gmessias/NodeClusteringAlgorithms/result...,5,0,5,5,0,Concluído com falhas
83,polblogs_benchmark_adaptive,densgnn_border,5,polblogs_border,densgnn,/home/gmessias/NodeClusteringAlgorithms/result...,0,0,0,0,5,Pendente/em execução
86,polblogs_benchmark_adaptive,densgnn_core,5,polblogs_core,densgnn,/home/gmessias/NodeClusteringAlgorithms/result...,0,0,0,0,5,Pendente/em execução
104,wiki_benchmark_adaptive,coane,5,wiki_benchmark_adaptive,coane,/home/gmessias/NodeClusteringAlgorithms/result...,5,0,5,5,0,Concluído com falhas


## Exportar

- `comparison_vs_densgnn.html`: tabela de métricas externas com cores, pronta para abrir no navegador.
- `internal/comparison_vs_densgnn.html` e `internal/comparison_vs_densgnn.tex`: segunda tabela, com as métricas internas. A pasta `internal/` também contém seus CSVs numéricos e de auditoria.
- `comparison_vs_densgnn.tex`: tabela LaTeX sem cores, com a maior média em negrito e a segunda maior distinta sublinhada; adicione `booktabs` e `graphicx` ao preâmbulo.
- `comparison_formatted.csv` e `comparison_relations.csv`: valores exibidos e classificação das comparações. CSV não armazena cores.
- `summary_numeric.csv`: médias e desvios nas unidades originais, contagens válidas e métricas solicitadas.
- `run_audit.csv`, `all_attempts.csv`, `selected_attempts.csv` e `load_issues.csv`: auditoria do retrato atual.

A exportação substitui somente esses arquivos de análise em `EXPORT_DIR`. Nenhum CSV de treinamento é alterado. Reexecute a leitura e a exportação para incorporar novas runs.

In [39]:
comparison.export_report(report, EXPORT_DIR, latex_style=LATEX_STYLE)
comparison.export_report(
    internal_report, EXPORT_DIR / "internal",
    table_label="tab:benchmark-densgnn-adaptive-internal",
    latex_style=LATEX_STYLE,
)
print(f"Exportações salvas em: {EXPORT_DIR}")
for path in sorted(EXPORT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(EXPORT_DIR)}")

Exportações salvas em: /home/gmessias/NodeClusteringAlgorithms/results/adaptive_benchmark_comparison
  all_attempts.csv
  comparison_formatted.csv
  comparison_relations.csv
  comparison_vs_densgnn.html
  comparison_vs_densgnn.tex
  internal/all_attempts.csv
  internal/comparison_formatted.csv
  internal/comparison_relations.csv
  internal/comparison_vs_densgnn.html
  internal/comparison_vs_densgnn.tex
  internal/load_issues.csv
  internal/run_audit.csv
  internal/selected_attempts.csv
  internal/summary_numeric.csv
  load_issues.csv
  run_audit.csv
  selected_attempts.csv
  summary_numeric.csv
  top3_subset/adaptive_ranks.csv
  top3_subset/cluster_size_violins/cluster_size_violins.pdf
  top3_subset/cluster_size_violins/cluster_size_violins.png
  top3_subset/cluster_size_violins/cluster_size_violins.svg
  top3_subset/cluster_size_violins/cluster_sizes.csv
  top3_subset/cluster_size_violins/coverage.csv
  top3_subset/cluster_size_violins/figure_caption.txt
  top3_subset/cluster_size_vio